In [50]:
import pandas as pd
import snakecase
from requests import get
from nba_api.stats.endpoints import boxscoretraditionalv2, boxscoreadvancedv2
from nba_api.stats.static import players, teams
from nba_api.stats.library.parameters import Season
from dataHub import dataHub
from sqlalchemy.dialects.postgresql.base import PGDialect
PGDialect._get_server_version_info = lambda *args: (9, 2)

dh = dataHub()
db_con = dh.db_connect('postgre')


In [62]:
bs_max_dt = (
    pd.read_sql_query("""
        SELECT MAX(ls.game_date) 
        FROM nba.team_box_score AS bs
        LEFT JOIN nba.league_game_schedule AS ls on bs.game_id = ls.game_id
    """, db_con)
    ['max'][0]
    .strftime('%Y-%m-%d')
)

game_ids = (
    pd.read_sql_query(f"""
        SELECT game_id, game_date
        FROM nba.league_game_schedule
        WHERE game_date > '{bs_max_dt}'
            AND game_date <= current_date
    """, db_con)
)

trad_adv_lst = ['player_', 'team_']

for game_id in game_ids['game_id'][0:4]:
    game_id = '00' + str(int(game_id))

    dfs = []
    bsa = boxscoreadvancedv2.BoxScoreAdvancedV2(game_id=game_id)
    if len(bsa.get_normalized_dict()['PlayerStats']) == 0:
        continue

    bst = boxscoretraditionalv2.BoxScoreTraditionalV2(game_id=game_id)
    
    for el in [0, 1]:

        bst_col_order = pd.read_sql_query(f"SELECT column_name FROM util.table_column_order WHERE table_name = '{trad_adv_lst[el]}box_score_traditional' ORDER BY column_order", db_con)['column_name'].to_list()
        bsa_col_order = pd.read_sql_query(f"SELECT column_name FROM util.table_column_order WHERE table_name = '{trad_adv_lst[el]}box_score_advanced' ORDER BY column_order", db_con)['column_name'].to_list()
        
        bst_df = (
            bst
            .get_data_frames()[el]
            .rename(snakecase.convert, axis='columns')
            .drop_duplicates()
            .assign(game_id = lambda x: x['game_id'].astype('int'))
            .assign(min = lambda x: [int(re.sub(r'\..*', '', el)) if el is not None else None for el in x['min']])
            [bst_col_order]
        )

        bsa_df = (
            bsa
            .get_data_frames()[el]
            .rename(snakecase.convert, axis='columns')
            .drop_duplicates()
            .assign(game_id = lambda x: x['game_id'].astype('int'))
            [bsa_col_order]
        )
        
        jn_cols = list(set(bst_df.columns) & set(bsa_df.columns))
        df = bst_df.merge(bsa_df, how='left', on=jn_cols)
        dfs.append(df)

    # dfs[0].to_sql('player_box_score', db_con, schema='nba', index=False, if_exists='append')
    # dfs[1].to_sql('team_box_score', db_con, schema='nba', index=False, if_exists='append')

dfs[0]

,game_id,team_id,team_abbreviation,player_id,player_name,start_position,comment,min,fgm,fga,...,tm_tov_pct,efg_pct,ts_pct,usg_pct,e_usg_pct,e_pace,pace,pace_per40,poss,pie
0,42300151,1610612747,LAL,1629060,Rui Hachimura,F,,31.0,2.0,4.0,...,14.3,0.625,0.717,0.090,0.092,95.71,95.74,79.79,62,0.049
1,42300151,1610612747,LAL,2544,LeBron James,F,,40.0,10.0,16.0,...,21.2,0.719,0.742,0.291,0.296,95.03,94.23,78.53,80,0.146
2,42300151,1610612747,LAL,203076,Anthony Davis,C,,44.0,12.0,23.0,...,5.9,0.522,0.593,0.309,0.314,93.37,93.30,77.75,87,0.171
3,42300151,1610612747,LAL,1630559,Austin Reaves,G,,36.0,5.0,9.0,...,0.0,0.667,0.689,0.117,0.117,93.88,92.84,77.37,71,0.092
4,42300151,1610612747,LAL,1626156,D'Angelo Russell,G,,41.0,6.0,20.0,...,4.2,0.325,0.325,0.244,0.250,92.70,92.88,77.40,80,0.017
5,42300151,1610612747,LAL,203915,Spencer Dinwiddie,,,13.0,0.0,0.0,...,0.0,0.000,0.000,0.000,0.000,104.66,104.29,86.91,28,0.036
6,42300151,1610612747,LAL,1627752,Taurean Prince,,,20.0,4.0,7.0,...,0.0,0.643,0.698,0.190,0.195,95.12,94.04,78.37,40,0.174
7,42300151,1610612747,LAL,1629637,Jaxson Hayes,,,4.0,0.0,0.0,...,0.0,0.000,0.000,0.000,0.000,91.11,93.08,77.57,9,0.056
8,42300151,1610612747,LAL,1629216,Gabe Vincent,,,7.0,0.0,0.0,...,50.0,0.000,0.000,0.067,0.068,88.89,90.39,75.32,14,-0.093
9,42300151,1610612747,LAL,1631108,Max Christie,,DNP - Coach's Decision,NaN,NaN,NaN,...,NaN,NaN,NaN,0.000,0.000,NaN,NaN,NaN,0,NaN


In [33]:
df = (
    df_bs
    .drop_duplicates()
    .assign(game_id = lambda x: x['game_id'].astype('int'))
    .assign(min = lambda x: [int(re.sub(r'\..*', '', el)) if el is not None else None for el in x['min']])
    .merge(df_ls, how='left', on='game_id')
    [['game_id', 'game_date', 'team_id', 'team_abbreviation', 'player_id', 'player_name',
       'start_position', 'comment', 'min', 'fgm', 'fga', 'fg_pct', 'fg3_m',
       'fg3_a', 'fg3_pct', 'ftm', 'fta', 'ft_pct', 'pts', 'oreb', 'dreb', 'reb',
       'ast', 'stl', 'blk', 'to', 'pf', 'plus_minus', 'e_off_rating', 'off_rating',
       'e_def_rating', 'def_rating', 'e_net_rating', 'net_rating', 'ast_pct',
       'ast_tov', 'ast_ratio', 'oreb_pct', 'dreb_pct', 'reb_pct', 'tm_tov_pct',
       'efg_pct', 'ts_pct', 'usg_pct', 'e_usg_pct', 'e_pace', 'pace',
       'pace_per40', 'poss', 'pie']]
)

df.head()

,game_id,game_date,team_id,team_abbreviation,player_id,player_name,start_position,comment,min,fgm,...,tm_tov_pct,efg_pct,ts_pct,usg_pct,e_usg_pct,e_pace,pace,pace_per40,poss,pie
0,22301193,2024-04-14,1610612743,DEN,1629008,Michael Porter Jr.,F,,24.0,5.0,...,11.1,0.500,0.500,0.230,0.231,99.26,97.91,81.59,50,0.117
1,22301193,2024-04-14,1610612743,DEN,203932,Aaron Gordon,F,,26.0,4.0,...,6.3,0.667,0.788,0.169,0.171,100.81,98.45,82.04,53,0.152
2,22301193,2024-04-14,1610612743,DEN,203999,Nikola Jokić,C,,30.0,7.0,...,19.0,0.636,0.631,0.203,0.207,102.48,101.35,84.46,65,0.204
3,22301193,2024-04-14,1610612743,DEN,203484,Kentavious Caldwell-Pope,G,,24.0,5.0,...,18.2,0.917,0.854,0.127,0.128,102.67,100.51,83.76,53,0.121
4,22301193,2024-04-14,1610612743,DEN,1627750,Jamal Murray,G,,24.0,7.0,...,9.1,0.607,0.666,0.295,0.296,99.26,97.91,81.59,50,0.185


In [3]:
for gid in gids:
    print(gid)
    gid = '00' + str(gid)

    dfs = []
    bsa = boxscoreadvancedv2.BoxScoreAdvancedV2(game_id=gid)
    if len(bsa.get_normalized_dict()['PlayerStats']) == 0:
        continue

    bst = boxscoretraditionalv2.BoxScoreTraditionalV2(game_id=gid)
    
    for el in [0, 1]:
        
        bst_df = bst.get_data_frames()[el]
        bsa_df = bsa.get_data_frames()[el]
        
        jn_cols = [el for el in bst_df.columns if el.endswith('ID')]
        dup_cols = list(set(bst_df.columns) & set(bsa_df.columns))
        drp_cols = [el for el in dup_cols if el not in jn_cols]
        
        df = (
            bst_df
            .drop(drp_cols, axis='columns')
            .merge(bsa_df, how='left', on=jn_cols)
            .rename(snakecase.convert, axis='columns')
        )
        
        dfs.append(df)

    dfs[0].to_sql('player_box_score', db_con, schema='nba', index=False, if_exists='append')
    dfs[1].to_sql('team_box_score', db_con, schema='nba', index=False, if_exists='append')

11300116
11300115
11300113
11300112
11300111
11300110
11300109
11300108
11300107
11300106
11300105
11300104
11300103
11300102
11300101
11300100
11300099
11300098
11300097
11300096
11300095
11300094
11300093
11300092
11300091
11300090
11300089
11300088
11300087
11300086
11300085
11300084
11300083
11300082
11300081
11300080
11300079
11300078
11300077
11300076
11300075
11300074
11300073
11300072
11300071
11300070
11300069
11300068
11300067
11300066
11300065
11300064
11300063
11300062
11300061
11300060
11300059
11300058
11300057
11300056
11300055
11300054
11300053
11300052
11300051
11300050
11300049
11300048
11300047
11300046
11300045
11300044
11300043
11300042
11300041
11300040
11300039
11300038
11300037
11300036
11300035
11300034
11300033
11300032
11300031
11300030
11300029
11300028
11300027
11300026
11300025
11300024
11300023
11300022
11300021
11300020
11300019
11300018
11300017
11300016
11300015
11300014
11300013
11300012
11300011
11300010
11300009
11300008
11300007
11300006
11300005
1

In [ ]:
request = get(f'https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule_week_tbds.json')

# Dataframe object to be added to
df = pd.DataFrame()

# Loop through month elements
for month in request.json()['lscd']:
    df_row = pd.DataFrame(month['mscd']['g'])[['gid', 'gdte', 'an', 'ac', 'htm', 'vtm', 'v', 'h']]

    # Obtain home and away team info
    for col in ['h', 'v']:
        df_col = pd.DataFrame([[el['tid'], el['ta']] for el in df_row[col]])
        df_col.columns = ['home_team_id', 'home_team_slug'] if col == 'h' else ['away_team_id', 'away_team_slug']
        df_row = pd.concat([df_row, df_col], axis = 1)

    # Rename columns
    df_row = df_row.drop(columns=['v', 'h'])
    df_row = df_row.rename(columns={'gid':'game_id', 'gdte':'game_date', 'an':'arena', 'ac':'city', 'htm':'home_team_time', 'vtm':'away_team_time'})

    # Collate data
    df = pd.concat([df, df_row], ignore_index=True)

%view df df

# Cast date columns to_date & Create new columns
df['game_date'] = [parse(el).date() for el in df['game_date']]
df['slug_season'] = Season.current_season
df['slug_matchup'] = df['home_team_slug'] + ' vs. ' + df['away_team_slug']
df['slug_team_winner'] = None
df['slug_team_loser'] = None

key_dates = read_sql_query('SELECT * FROM util.key_dates', db_con)
df = sqldf("""
    SELECT 
        key_dates.season_type AS type_season, 
        df.*
    FROM df
    LEFT JOIN key_dates ON df.game_date >= key_dates.begin_date AND df.game_date <= key_dates.end_date
""", locals())[col_order]

# Write to database
df.to_sql('league_game_schedule', db_con, schema='nba', index=False, if_exists='append')
# return df
print('current_game_schedule has been updated\n\n')